### plotting

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import pandas as pd

# Load data
df = pd.read_csv('melted_data_for_fig.csv') 


sns.set(rc={'axes.facecolor':'whitesmoke', 'figure.facecolor':'whitesmoke',"grid.color": "silver"})



# Manual x positions for each strain — adjust to control spacing to control figure appearence
positions = np.array([2.302, 2.93, 3.618, 4.305, 5.0])
# positions = np.array([2.302, 2.90, 3.544, 4.305, 5.0])

width = 0.5

vals = [df_plot.loc[df_plot["strain"] == s, "value"].dropna().to_numpy()
        for s in order]

# Create figure
f, ay = plt.subplots(figsize=(14, 5))
sns.set(style="darkgrid", rc={'axes.facecolor': 'whitesmoke', 'figure.facecolor': 'whitesmoke', "grid.color": "silver"})

ay.grid(False)
ay.grid(axis="y", linestyle="-", linewidth=0.8)

# Axis labels and limits
ay.set_xlabel('Strain', fontsize=16)
ay.set_ylabel('Abs590', fontsize=16)
ay.set_ylim(-0.03, 0.9)
ay.set_yticks([0.2, 0.4, 0.6, 0.8], minor=False)
ay.tick_params(axis='both', which='major', labelsize=14)

# Boxplot
bp = ay.boxplot(
    vals,
    positions=positions,
    widths=width,
    patch_artist=True,
    showfliers=False,
    showcaps=False,
    whiskerprops=dict(linewidth=1.2, visible=True),
    medianprops=dict(linewidth=2, color="black"),
    boxprops=dict(linewidth=1.1, edgecolor="black"),
)

for patch in bp["boxes"]:
    patch.set_facecolor("gainsboro")
    patch.set_alpha(1.0)

# Overlay individual points
rng = np.random.default_rng(7)
for x, y in zip(positions, vals):
    if len(y) == 0:
        continue
    jitter = rng.normal(0, 0.06, size=len(y))
    ay.scatter(
        np.full(len(y), x) + jitter,
        y,
        s=150,
        color="rebeccapurple",
        alpha=0.6,
        zorder=3,
    )

# Axes formatting
ay.set_xticks(positions)
ay.set_xticklabels(order, fontsize=14)
ay.set_xlim(positions[0] - 0.6, positions[-1] + 0.6)

sns.despine(top=True, bottom=True, right=True)

ay.grid(False)
ay.grid(axis="y", linestyle="-", linewidth=0.8)


plt.savefig('biofilm_figure_for_fig4_2.png', dpi=300, bbox_inches='tight')
plt.show()

In [ ]:
print(order, len(order))

# stats

In [ ]:
# --- Statistical analysis for Fig 45 biofilm ---
# 1. Average technical reps within each biological replicate
# 2. Randomized block ANOVA (strain + replicate block)
# 3. Dunnett's for all vs WT, plus separate t-test for fj89 vs fj68

import pandas as pd
from statsmodels.formula.api import ols
import statsmodels.api as sm
from scipy.stats import dunnett, ttest_ind

df = pd.read_csv('melted_data_for_fig5.csv')
bio = df[(df['expt'] == 'Biofilm') & (df['strain'] != 'fjxx')].copy()

# Average technical replicates
bio_means = bio.groupby(['strain', 'replicate'])['value'].mean().reset_index()

# Randomized block ANOVA
model = ols('value ~ C(strain) + C(replicate)', data=bio_means).fit()
anova_table = sm.stats.anova_lm(model, typ=2)
print("--- ANOVA (Biofilm, Fig 5) ---")
print(anova_table)

# Dunnett's: all vs wt
control = bio_means[bio_means['strain'] == 'wt']['value'].values
compare_strains = ['fj28', 'fj68', 'fj89', 'fj66']
treatments = {s: bio_means[bio_means['strain'] == s]['value'].values
              for s in compare_strains}

result = dunnett(*treatments.values(), control=control)
print("\n--- Dunnett's test vs WT ---")
for strain, p in zip(treatments.keys(), result.pvalue):
    print(f"wt vs {strain}: p = {p:.4f}")

# Additional comparison: fj89 vs fj68
fj89 = bio_means[bio_means['strain'] == 'fj89']['value'].values
fj68 = bio_means[bio_means['strain'] == 'fj68']['value'].values
t, p = ttest_ind(fj89, fj68, equal_var=False)
print(f"\n--- fj89 vs fj68 (Welch's t-test) ---")
print(f"t = {t:.4f}, p = {p:.4f}")